# Trimming and Filtering Reads with fastp

_BioNexus Hub - bionexushub.com · Companion to the lesson._

Uses the **Python** runtime (default). Lines starting with `!` run shell commands. Run each cell with Shift+Enter.


## 1. Install fastp


In [ ]:
!apt-get -qq update && apt-get -qq install -y fastp
!fastp --version


## 2. Make example reads (quality drop + adapter read-through in ~30%)


In [ ]:
import random
random.seed(1)
ADAPTER = "AGATCGGAAGAGCACACGTCTGAACTCCAGTCA"   # common Illumina adapter
def qual(n):
    out = []
    for i in range(n):
        q = int(40 - i*0.22 + random.gauss(0, 2)); q = max(2, min(40, q))
        out.append(chr(33 + q))
    return ''.join(out)

L = 100
with open('reads.fastq', 'w') as f:
    for i in range(3000):
        seq = ''.join(random.choice('ACGT') for _ in range(L))
        if random.random() < 0.3:               # simulate adapter read-through
            cut = random.randint(40, 75)
            seq = (seq[:cut] + ADAPTER + seq)[:L]
        f.write(f'@read{i}\n{seq}\n+\n{qual(L)}\n')

!head -4 reads.fastq


## 3. Trim with fastp
Two fastp defaults to know: for single-end reads, adapter auto-detection can miss adapters, and quality trimming of read ends is **off** unless you ask. So we name the standard Illumina adapter and switch on `--cut_right` (cut the read where quality drops). Watch the summary it prints.

In [ ]:
!fastp -i reads.fastq -o reads.trimmed.fastq \
      --adapter_sequence AGATCGGAAGAGCACACGTCTGAACTCCAGTCA \
      --cut_right \
      -h fastp.html -j fastp.json

## 4. Before vs after


In [ ]:
!echo "Before: $(( $(wc -l < reads.fastq)/4 )) reads"
!echo "After:  $(( $(wc -l < reads.trimmed.fastq)/4 )) reads"


In [ ]:
import json
r = json.load(open("fastp.json"))
print("Q20 before:", round(r["summary"]["before_filtering"]["q20_rate"], 3))
print("Q20 after: ", round(r["summary"]["after_filtering"]["q20_rate"], 3))
print("reads with adapter trimmed:", r["adapter_cutting"]["adapter_trimmed_reads"])

### See the full report
Download `fastp.html` from the file browser on the left (folder icon) to see the before-and-after graphs.
